# Scenario 1 — Hello-World Workflow through the Knowledge Graph

This notebook demonstrates the core claim of the KAPPS Semantic Middleware: workflows are discovered and invoked **entirely through the knowledge graph**, with no hard-coded service registries or configuration files. A "hello-world" middleware wraps a simple function, registers a Service/Capability/Workflow structure in GraphDB on startup, and advertises a reachable endpoint. A second middleware, given only an Operation IRI, resolves Operation → Capability → Workflow → endpoint purely by querying the graph, invokes it over HTTP, and records the outcome back on the Operation (decision provenance per R12). On shutdown, the first middleware deregisters (removes reachability) but the individuals persist for audit.

**ADR 0010 self-contained promise:** This example connects to a GraphDB from environment variables (`GRAPHDB_URL`, `GRAPHDB_USERNAME`, `GRAPHDB_PASSWORD`, `GRAPHDB_REPOSITORY`), **clears a dedicated test repository**, loads exactly the ontology it needs (`svc:` module + demo ontology), seeds the starting state, then runs. It is fully reproducible and never touches production state.

In [1]:
import os
import sys
import threading
import time
from pathlib import Path

import httpx
import uvicorn
from rdflib.namespace import RDF

from graph_db_interface import GraphDB
from kapps_ogm import OGM
from kapps_semantic_middleware import SemanticMiddleware
from kapps_semantic_middleware.registration import (
    mint_capability_iri,
    mint_workflow_iri,
)
from kapps_semantic_middleware.vocabulary import SVC

# seed.py sits beside this notebook in examples/. Make it importable regardless of
# the working directory the notebook is launched from.
for _cand in (Path.cwd(), Path.cwd() / "examples"):
    if (_cand / "seed.py").exists():
        sys.path.insert(0, str(_cand))
        break
import seed  # noqa: E402

# Connect to GraphDB from environment.
db = GraphDB.from_env()
print(f"Connected to GraphDB repository: {os.getenv('GRAPHDB_REPOSITORY')}")

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


Connected to GraphDB repository: Tests


## Step 1 — Seed a Clean Repository

We clear the repository's default graph and load exactly the ontologies this scenario needs: the `svc:` module (Service/Capability/Workflow vocabulary) and the demo ontology (scenario-specific classes). Then we create the two resource individuals: `hello_resource` and `planner_resource`. The Operation is created later, after the hello-world middleware has registered its capability instance.

In [2]:
seed.seed_scenario1(db)

hello_exists = db.triple_exists((seed.HELLO_RESOURCE, RDF.type, seed.HELLO_RESOURCE_CLASS))
print(f"Hello resource instantiated:   {hello_exists}")
print(
    f"Planner resource instantiated: "
    f"{db.triple_exists((seed.PLANNER_RESOURCE, RDF.type, seed.PLANNER_RESOURCE_CLASS))}"
)

Hello resource instantiated:   True
Planner resource instantiated: True


## Step 2 — Start the Hello-World Middleware

We wrap a trivial `hello_world()` function (imported from `handlers.py`) with `@workflow`. It lives in a module rather than a notebook cell because `@workflow` type-checks the function by reading its module source, and a function defined in a Jupyter cell has none. The middleware runs in "resource" mode, representing `hello_resource` and exposing its capabilities. On startup it registers a Service/Capability/Workflow structure in the graph and starts an HTTP server advertising the workflow's endpoint. We run uvicorn in a daemon thread and poll until `server.started` is True.

In [3]:
from handlers import hello_world  # noqa: E402

HELLO_PORT = 8993

mw1 = SemanticMiddleware(
    mode="resource",
    resource_iri=seed.HELLO_RESOURCE,
    service_class=seed.HELLO_SERVICE_CLASS,
    ogm=OGM(db=db),
    host="127.0.0.1",
    port=HELLO_PORT,
)
mw1.workflow(
    capability_class=seed.HELLO_CAPABILITY_CLASS,
    workflow_class=seed.HELLO_WORKFLOW_CLASS,
)(hello_world)

config = uvicorn.Config(mw1.app, host="127.0.0.1", port=HELLO_PORT, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

t0 = time.time()
while not server.started and time.time() - t0 < 30:
    time.sleep(0.05)
if not server.started:
    raise RuntimeError("server did not start in time")

print(f"Hello-world middleware started on port {HELLO_PORT} (registered on startup)")

Hello-world middleware started on port 8993 (registered on startup)


### 🔎 The middleware's live REST API (Swagger UI)

The middleware is now a running FastAPI server. Its interactive Swagger UI is live at
**http://127.0.0.1:8993/docs** — open it in a browser to see the auto-generated
`POST /workflows/hello_world/execute` endpoint (and `/health`) and try them out. The server
stays up until the shutdown cell (Step 6), so you can explore it while stepping through the
notebook. (On a remote host, forward port 8993 as well as the Jupyter port.)

In [4]:
print("Swagger UI:  http://127.0.0.1:8993/docs")
print("OpenAPI:     http://127.0.0.1:8993/openapi.json")
print("Registered routes:", [r.path for r in mw1.app.routes if "workflows" in r.path])

Swagger UI:  http://127.0.0.1:8993/docs
OpenAPI:     http://127.0.0.1:8993/openapi.json
Registered routes: ['/workflows/hello_world/execute', '/workflows/hello_world/execute_background', '/workflows/hello_world/description', '/workflows/hello_world/interrupt']


## Step 3 — Inspect What Registration Wrote

Registration wrote the full Service/Capability/Workflow structure plus reachability triples. We compute the IRIs deterministically using the same `mint_*` helpers the middleware uses, then query the graph to confirm the structural triples and read back the advertised address and endpoint. This is the "advertised state" other agents discover.

In [5]:
service_iri = seed.HELLO_RESOURCE + "_service"
cap_instance = mint_capability_iri(seed.HELLO_RESOURCE, "hello_world")
wf_instance = mint_workflow_iri(service_iri, "hello_world")

print(f"Service IRI:    {service_iri}")
print(f"Capability IRI: {cap_instance}")
print(f"Workflow IRI:   {wf_instance}\n")

assert db.triple_exists((service_iri, RDF.type, seed.HELLO_SERVICE_CLASS))
assert db.triple_exists((service_iri, SVC.isServiceOf, seed.HELLO_RESOURCE))
assert db.triple_exists((cap_instance, SVC.realizedByWorkflow, wf_instance))
assert db.triple_exists((wf_instance, SVC.isWorkflowOf, service_iri))
print("Structural triples present (isServiceOf, realizedByWorkflow, isWorkflowOf).")
print("Note: only the instance-owned direction of each inverse is materialized (ADR 0008);")
print("the container-side inverses (hasService/hasWorkflow) are OWL-inferable and not written.")

address_triples = list(db.triples_get(sub=service_iri, pred=SVC.address))
endpoint_triples = list(db.triples_get(sub=wf_instance, pred=SVC.endpoint))
print(f"Service address advertised:  {address_triples[0][2] if address_triples else 'NONE'}")
print(f"Workflow endpoint advertised: {endpoint_triples[0][2] if endpoint_triples else 'NONE'}")

Service IRI:    https://example.org/kapps-demo#hello_resource_service
Capability IRI: https://example.org/kapps-demo#hello_resource_capability_hello_world
Workflow IRI:   https://example.org/kapps-demo#hello_resource_service_workflow_hello_world

Structural triples present (isServiceOf, realizedByWorkflow, isWorkflowOf).
Note: only the instance-owned direction of each inverse is materialized (ADR 0008);
the container-side inverses (hasService/hasWorkflow) are OWL-inferable and not written.
Service address advertised:  http://127.0.0.1:8993
Workflow endpoint advertised: http://127.0.0.1:8993/workflows/hello_world/execute


## Step 4 — Resolve and Execute from a Second Middleware

Now a planner agent wants to invoke the hello-world capability. We create an Operation individual that implements the capability. A second middleware resolves Operation → Capability → Workflow → endpoint purely by querying the graph, invokes it over HTTP, and returns the result. We use top-level `await` here because we are running inside a Jupyter kernel with an existing event loop (`asyncio.run` would fail).

In [6]:
seed.create_operation(db, seed.HELLO_OPERATION, cap_instance)
print(f"Operation created: {seed.HELLO_OPERATION}")
print(f"  implementsCapability -> {cap_instance}\n")

mw2 = SemanticMiddleware(
    mode="resource",
    resource_iri=seed.PLANNER_RESOURCE,
    service_class=seed.PLANNER_SERVICE_CLASS,
    ogm=OGM(db=db),
    host="127.0.0.1",
    port=8994,
)

result = await mw2.execute(seed.HELLO_OPERATION)  # noqa: F704  (top-level await; Jupyter)

print("Execution result:")
print(f"  success:  {result['success']}")
print(f"  result:   {result['result']}")
print(f"  workflow: {result['workflow']}")

INFO:httpx:HTTP Request: POST http://127.0.0.1:8993/workflows/hello_world/execute "HTTP/1.1 200 OK"


Operation created: https://example.org/kapps-demo#helloWorldOperation_1
  implementsCapability -> https://example.org/kapps-demo#hello_resource_capability_hello_world



Execution result:
  success:  True
  result:   hello world
  workflow: https://example.org/kapps-demo#hello_resource_service_workflow_hello_world


## Step 5 — The Decision is Now Traceable in the Graph (R12)

Per requirement R12, the middleware writes decision provenance back onto the Operation after execution: which Workflow executed it (`executedByWorkflow`) and when (`executionTimestamp`). Whether it succeeded is carried by the Operation's terminal status (`done`/`failed`) rather than a separate `executionSuccess` boolean (ADR 0009). Every runtime decision is auditable directly from the knowledge graph.

In [ ]:
executed_by = list(db.triples_get(sub=seed.HELLO_OPERATION, pred=SVC.executedByWorkflow))
timestamp = list(db.triples_get(sub=seed.HELLO_OPERATION, pred=SVC.executionTimestamp))

print(f"Decision provenance recorded on {seed.HELLO_OPERATION}:")
print(f"  executedByWorkflow: {executed_by[0][2] if executed_by else 'NONE'}")
print(f"  executionTimestamp: {timestamp[0][2] if timestamp else 'NONE'}")

## Step 6 — Shutdown and Deregistration

On shutdown, the hello-world middleware deregisters by removing the reachability triples (address and endpoint) but **preserves the individuals** (Service, Capability, Workflow) for audit. We stop the server and verify address/endpoint are gone while the workflow's `rdf:type` persists.

In [8]:
server.should_exit = True
thread.join(timeout=20)
time.sleep(0.5)

address_after = list(db.triples_get(sub=service_iri, pred=SVC.address))
endpoint_after = list(db.triples_get(sub=wf_instance, pred=SVC.endpoint))
print("After shutdown:")
print(f"  Service address removed:  {len(address_after) == 0}")
print(f"  Workflow endpoint removed: {len(endpoint_after) == 0}")
print(
    f"  Workflow individual preserved (rdf:type): "
    f"{db.triple_exists((wf_instance, RDF.type, seed.HELLO_WORKFLOW_CLASS))}"
)

After shutdown:
  Service address removed:  True
  Workflow endpoint removed: True
  Workflow individual preserved (rdf:type): True


## Recap

This scenario demonstrates three core claims of the KAPPS architecture:

1. **KG as authoritative runtime state** — the graph holds the complete Service/Capability/Workflow registry; agents query the graph rather than any external discovery mechanism.
2. **Capability-mediated discovery** — the planner never knew the workflow's URL or name; it specified an Operation implementing a Capability, and the middleware resolved Capability → Workflow → endpoint through the graph.
3. **Decision provenance (R12)** — every execution writes its outcome back to the graph, and deregistration-on-shutdown preserves individuals while removing reachability, so the graph is a complete historical record of what was available, what was invoked, and what happened.